In [103]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# Load the data
df = pd.read_csv('COVID-19_Report.csv', low_memory=False)

In [5]:
# Establish the variable groups by type

numeric_variables = [
    'Action Obligation',
    'Base and All Options Value (Total Contract Value)'
]

date_variables = [
    'Solicitation Date',
    'Completion Date',
    'Period of Performance Start Date',
    'Date Signed'
]

text_variables = [
    'Description of Requirement',
    'Product or Service Description'
]

categorical_variables = [
    'Domestic or Foreign Entity',
    'Vendor Address State Name',
    'Solicitation Procedures',
    'Extent Competed',
    'NAICS Code',
    'Type of Contract',
    'Product or Service Code',
    "Contracting Officer's Business Size Determination",
    'Contracting Office Name',
    'Award Type',
    'Funding Department Name',
    'Other Than Full and Open Competition',
    'Fair Opportunity/Limited Sources',
    'Type of Set Aside'
]

selected_variables = (
    numeric_variables
    + date_variables
    + text_variables
    + categorical_variables
)

print(f"Number of selected variables: {len(selected_variables)}")

Number of selected variables: 22


In [7]:
print(f"Original dataset shape: {df.shape}")

Original dataset shape: (150000, 176)


In [9]:
# Verify all required variables exist

missing_columns = [
    column
    for column in selected_variables
    if column not in df.columns
]

if missing_columns:
    print("Missing required columns:")
    for column in missing_columns:
        print(f"  - {column}")
else:
    print("All 22 selected variables are present.")

All 22 selected variables are present.


In [11]:
# Create modeling dataframe

model_data = df[selected_variables].copy()

print(f"Modeling dataset shape: {model_data.shape}")

Modeling dataset shape: (150000, 22)


In [13]:
model_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 22 columns):
 #   Column                                             Non-Null Count   Dtype 
---  ------                                             --------------   ----- 
 0   Action Obligation                                  150000 non-null  object
 1   Base and All Options Value (Total Contract Value)  150000 non-null  object
 2   Solicitation Date                                  63162 non-null   object
 3   Completion Date                                    144712 non-null  object
 4   Period of Performance Start Date                   149997 non-null  object
 5   Date Signed                                        150000 non-null  object
 6   Description of Requirement                         149978 non-null  object
 7   Product or Service Description                     150000 non-null  object
 8   Domestic or Foreign Entity                         144927 non-null  object
 9   Vend

In [15]:
# Reassess missingness

missing_summary_before = pd.DataFrame({
    'Missing Count': model_data.isna().sum(),
    'Missing Percent': (
        model_data.isna().mean() * 100
    )
})

missing_summary_before = (
    missing_summary_before
    .sort_values(
        'Missing Percent',
        ascending=False
    )
)

print(
    missing_summary_before
    .round(2)
    .to_string()
)

                                                   Missing Count  Missing Percent
Other Than Full and Open Competition                      115196            76.80
Fair Opportunity/Limited Sources                          107795            71.86
Type of Set Aside                                         100038            66.69
Solicitation Date                                          86838            57.89
Completion Date                                             5288             3.53
Domestic or Foreign Entity                                  5073             3.38
Vendor Address State Name                                   1886             1.26
Solicitation Procedures                                     1450             0.97
Extent Competed                                             1370             0.91
Description of Requirement                                    22             0.01
Period of Performance Start Date                               3             0.00
Product or Servi

In [17]:
# Examine missingness in categorical variables 

categorical_missing_before = pd.DataFrame({
    'Missing Count': model_data[categorical_variables].isna().sum(),
    'Missing Percent': (
        model_data[categorical_variables].isna().mean() * 100
    )
})

categorical_missing_before = (
    categorical_missing_before
    .sort_values(
        'Missing Percent',
        ascending=False
    )
)

print(
    categorical_missing_before
    .round(2)
    .to_string()
)

                                                   Missing Count  Missing Percent
Other Than Full and Open Competition                      115196            76.80
Fair Opportunity/Limited Sources                          107795            71.86
Type of Set Aside                                         100038            66.69
Domestic or Foreign Entity                                  5073             3.38
Vendor Address State Name                                   1886             1.26
Solicitation Procedures                                     1450             0.97
Extent Competed                                             1370             0.91
NAICS Code                                                     0             0.00
Type of Contract                                               0             0.00
Product or Service Code                                        0             0.00
Contracting Officer's Business Size Determination              0             0.00
Contracting Offi

In [19]:
# Create explicit missing category for each categorical variable

for column in categorical_variables:
    model_data[column] = (
        model_data[column]
        .fillna('Missing')
    )

In [21]:
# Validate that missing data was treated correctly 

categorical_missing_after = pd.DataFrame({
    'Missing Count': model_data[categorical_variables].isna().sum(),
    'Missing Percent': (
        model_data[categorical_variables].isna().mean() * 100
    )
})

print(
    categorical_missing_after
    .round(2)
    .to_string()
)

                                                   Missing Count  Missing Percent
Domestic or Foreign Entity                                     0              0.0
Vendor Address State Name                                      0              0.0
Solicitation Procedures                                        0              0.0
Extent Competed                                                0              0.0
NAICS Code                                                     0              0.0
Type of Contract                                               0              0.0
Product or Service Code                                        0              0.0
Contracting Officer's Business Size Determination              0              0.0
Contracting Office Name                                        0              0.0
Award Type                                                     0              0.0
Funding Department Name                                        0              0.0
Other Than Full 

In [23]:
# Count explicit missing categories

explicit_missing_summary = []

for column in categorical_variables:

    missing_count = (
        model_data[column]
        .eq('Missing')
        .sum()
    )

    explicit_missing_summary.append({
        'Variable': column,
        'Missing Category Count': missing_count,
        'Missing Category Percent':
            (missing_count / len(model_data)) * 100
    })

explicit_missing_summary = pd.DataFrame(
    explicit_missing_summary
).sort_values(
    'Missing Category Percent',
    ascending=False
)

print(
    explicit_missing_summary
    .round(2)
    .to_string(index=False)
)

                                         Variable  Missing Category Count  Missing Category Percent
             Other Than Full and Open Competition                  115196                     76.80
                 Fair Opportunity/Limited Sources                  107795                     71.86
                                Type of Set Aside                  100038                     66.69
                       Domestic or Foreign Entity                    5073                      3.38
                        Vendor Address State Name                    1886                      1.26
                          Solicitation Procedures                    1450                      0.97
                                  Extent Competed                    1370                      0.91
                                       NAICS Code                       0                      0.00
                                 Type of Contract                       0                      0.00


In [25]:
# Examine missingness in date variables

date_missing_summary = pd.DataFrame({
    'Missing Count':
        model_data[date_variables].isna().sum(),

    'Missing Percent':
        model_data[date_variables].isna().mean() * 100
})

date_missing_summary = (
    date_missing_summary
    .sort_values(
        'Missing Percent',
        ascending=False
    )
)

print(
    date_missing_summary
    .round(2)
    .to_string()
)

                                  Missing Count  Missing Percent
Solicitation Date                         86838            57.89
Completion Date                            5288             3.53
Period of Performance Start Date              3             0.00
Date Signed                                   0             0.00


In [27]:
# Examine missingness in text variables

text_missing_summary = pd.DataFrame({
    'Missing Count':
        model_data[text_variables].isna().sum(),

    'Missing Percent':
        model_data[text_variables].isna().mean() * 100
})

print(
    text_missing_summary
    .round(4)
    .to_string()
)

                                Missing Count  Missing Percent
Description of Requirement                 22           0.0147
Product or Service Description              0           0.0000


In [29]:
# Recheck data

print("=" * 70)
print("MODEL DATA PREPARATION — INITIAL CHECKPOINT")
print("=" * 70)

print(f"\nRows: {model_data.shape[0]:,}")
print(f"Columns: {model_data.shape[1]}")

print("\nRemaining pandas missing values by variable:")
remaining_missing = model_data.isna().sum()
print(
    remaining_missing[
        remaining_missing > 0
    ].sort_values(ascending=False)
)

print("\nCategorical pandas missing values:")
print(
    model_data[categorical_variables]
    .isna()
    .sum()
    .sum()
)

print("\nExplicit categorical 'Missing' values:")
print(
    model_data[categorical_variables]
    .eq('Missing')
    .sum()
    .sum()
)

MODEL DATA PREPARATION — INITIAL CHECKPOINT

Rows: 150,000
Columns: 22

Remaining pandas missing values by variable:
Solicitation Date                   86838
Completion Date                      5288
Description of Requirement             22
Period of Performance Start Date        3
dtype: int64

Categorical pandas missing values:
0

Explicit categorical 'Missing' values:
332808


In [31]:
# Convert dates 

for column in date_variables:
    model_data[column] = pd.to_datetime(
        model_data[column],
        format='%m/%d/%Y',
        errors='coerce'
    )

In [33]:
# Validate

date_validation = pd.DataFrame({
    'Non-Missing': model_data[date_variables].notna().sum(),
    'Missing': model_data[date_variables].isna().sum(),
    'Missing Percent':
        model_data[date_variables].isna().mean() * 100,
    'Minimum': model_data[date_variables].min(),
    'Maximum': model_data[date_variables].max()
})

print(
    date_validation
    .round(2)
    .to_string()
)

                                  Non-Missing  Missing  Missing Percent    Minimum    Maximum
Solicitation Date                       63162    86838            57.89 2000-07-16 2022-09-29
Completion Date                        144706     5294             3.53 2008-12-31 2109-05-31
Period of Performance Start Date       149997        3             0.00 1981-01-25 2024-01-31
Date Signed                            150000        0             0.00 2020-03-13 2023-05-23


In [35]:
# Engineer two new temporal variables: Performance Duration Days and Solicitation to Signing Days

model_data['Performance Duration Days'] = (
    model_data['Completion Date']
    - model_data['Period of Performance Start Date']
).dt.days

model_data['Solicitation to Signing Days'] = (
    model_data['Date Signed']
    - model_data['Solicitation Date']
).dt.days

In [37]:
# Validate 

interval_variables = [
    'Performance Duration Days',
    'Solicitation to Signing Days'
]

for column in interval_variables:

    series = model_data[column]

    print(f"\n{'=' * 70}")
    print(column)
    print('=' * 70)

    print(f"Valid:    {series.notna().sum():,}")
    print(f"Missing:  {series.isna().sum():,}")
    print(f"Negative: {(series < 0).sum():,}")
    print(f"Zero:     {(series == 0).sum():,}")
    print(f"Positive: {(series > 0).sum():,}")
    print(f"Minimum:  {series.min():,.0f}")
    print(f"Median:   {series.median():,.0f}")
    print(f"Maximum:  {series.max():,.0f}")


Performance Duration Days
Valid:    144,703
Missing:  5,297
Negative: 20
Zero:     11,469
Positive: 133,214
Minimum:  -393
Median:   30
Maximum:  33,237

Solicitation to Signing Days
Valid:    63,162
Missing:  86,838
Negative: 0
Zero:     20,366
Positive: 42,796
Minimum:  0
Median:   60
Maximum:  7,913


In [39]:
# Create missingness indicators in temporal variables missing data

model_data['Solicitation Date Missing'] = (
    model_data['Solicitation Date']
    .isna()
    .astype(int)
)

model_data['Completion Date Missing'] = (
    model_data['Completion Date']
    .isna()
    .astype(int)
)

model_data['Performance Start Date Missing'] = (
    model_data['Period of Performance Start Date']
    .isna()
    .astype(int)
)

In [41]:
print("Temporal missingness indicators:")
print()

for column in [
    'Solicitation Date Missing',
    'Completion Date Missing',
    'Performance Start Date Missing'
]:
    print(
        f"{column}: "
        f"{model_data[column].sum():,}"
    )

Temporal missingness indicators:

Solicitation Date Missing: 86,838
Completion Date Missing: 5,294
Performance Start Date Missing: 3


In [43]:
# Analysis of distrbution of elapsed time based on earliest Date Signed (March 13, 2020)

signing_reference_date = model_data['Date Signed'].min()

model_data['Signing Elapsed Days'] = (
    model_data['Date Signed']
    - signing_reference_date
).dt.days

print(f"Signing reference date: {signing_reference_date.date()}")

print(
    model_data['Signing Elapsed Days']
    .describe()
)

Signing reference date: 2020-03-13
count    150000.000000
mean        273.611933
std         244.936275
min           0.000000
25%          75.000000
50%         187.000000
75%         419.000000
max        1166.000000
Name: Signing Elapsed Days, dtype: float64


In [47]:
# Calendar date feature engineering using the earliest Date Signed as reference point

signing_reference_date = model_data['Date Signed'].min()

model_data['Signing Elapsed Days'] = (
    model_data['Date Signed']
    - signing_reference_date
).dt.days

signing_month = model_data['Date Signed'].dt.month

model_data['Signing Month Sin'] = np.sin(
    2 * np.pi * signing_month / 12
)

model_data['Signing Month Cos'] = np.cos(
    2 * np.pi * signing_month / 12
)

print(
    f"Signing reference date: "
    f"{signing_reference_date.date()}"
)

print("\nEngineered calendar features:")
print(
    model_data[
        [
            'Date Signed',
            'Signing Elapsed Days',
            'Signing Month Sin',
            'Signing Month Cos'
        ]
    ]
    .head(10)
    .to_string(index=False)
)

Signing reference date: 2020-03-13

Engineered calendar features:
Date Signed  Signing Elapsed Days  Signing Month Sin  Signing Month Cos
 2020-04-13                    31           0.866025      -5.000000e-01
 2020-07-01                   110          -0.500000      -8.660254e-01
 2020-04-22                    40           0.866025      -5.000000e-01
 2020-09-21                   192          -1.000000      -1.836970e-16
 2020-09-02                   173          -1.000000      -1.836970e-16
 2020-04-02                    20           0.866025      -5.000000e-01
 2020-04-08                    26           0.866025      -5.000000e-01
 2020-04-21                    39           0.866025      -5.000000e-01
 2020-04-28                    46           0.866025      -5.000000e-01
 2020-05-06                    54           0.500000      -8.660254e-01


In [49]:
# Validation

temporal_features = [
    'Signing Elapsed Days',
    'Signing Month Sin',
    'Signing Month Cos',
    'Performance Duration Days',
    'Solicitation to Signing Days',
    'Solicitation Date Missing',
    'Completion Date Missing',
    'Performance Start Date Missing'
]

temporal_feature_summary = pd.DataFrame({
    'Missing': model_data[temporal_features].isna().sum(),
    'Minimum': model_data[temporal_features].min(),
    'Maximum': model_data[temporal_features].max()
})

print(
    temporal_feature_summary
    .round(4)
    .to_string()
)

                                Missing  Minimum  Maximum
Signing Elapsed Days                  0      0.0   1166.0
Signing Month Sin                     0     -1.0      1.0
Signing Month Cos                     0     -1.0      1.0
Performance Duration Days          5297   -393.0  33237.0
Solicitation to Signing Days      86838      0.0   7913.0
Solicitation Date Missing             0      0.0      1.0
Completion Date Missing               0      0.0      1.0
Performance Start Date Missing        0      0.0      1.0


In [51]:
# Set reference date as March 13, 2020 since this was confirmed to be earliest Date Signed

SIGNING_REFERENCE_DATE = pd.Timestamp('2020-03-13')

In [53]:
# Examine monetary variables 

for column in numeric_variables:
    model_data[column] = pd.to_numeric(
        model_data[column],
        errors='coerce'
    )

    print(f"\n{'=' * 70}")
    print(column)
    print('=' * 70)

    print(f"Missing:  {model_data[column].isna().sum():,}")
    print(f"Negative: {(model_data[column] < 0).sum():,}")
    print(f"Zero:     {(model_data[column] == 0).sum():,}")
    print(f"Positive: {(model_data[column] > 0).sum():,}")
    print(f"Minimum:  {model_data[column].min():,.2f}")
    print(f"Maximum:  {model_data[column].max():,.2f}")


Action Obligation
Missing:  150,000
Negative: 0
Zero:     0
Positive: 0
Minimum:  nan
Maximum:  nan

Base and All Options Value (Total Contract Value)
Missing:  150,000
Negative: 0
Zero:     0
Positive: 0
Minimum:  nan
Maximum:  nan


In [55]:
# Resotre the original monetary variables

model_data['Action Obligation'] = df['Action Obligation'].copy()

model_data[
    'Base and All Options Value (Total Contract Value)'
] = df[
    'Base and All Options Value (Total Contract Value)'
].copy()

In [57]:
# Examine data

print(model_data[numeric_variables].head(10))
print()
print(model_data[numeric_variables].dtypes)

  Action Obligation Base and All Options Value (Total Contract Value)
0          $701.19                                           $701.19 
1      $100,000.00                                       $100,000.00 
2      $400,000.00                                       $400,000.00 
3       $28,511.95                                        $28,511.95 
4       $33,603.43                                        $33,603.43 
5       $48,314.00                                        $48,314.00 
6       $55,056.00                                        $55,056.00 
7       $37,296.00                                        $37,296.00 
8       $47,952.00                                        $47,952.00 
9       $47,952.00                                        $47,952.00 

Action Obligation                                    object
Base and All Options Value (Total Contract Value)    object
dtype: object


In [59]:
for column in numeric_variables:

    print(f"\n{'=' * 70}")
    print(column)
    print('=' * 70)

    print("First 20 raw values:")
    print(
        model_data[column]
        .head(20)
        .tolist()
    )

    print("\nPython value types:")
    print(
        model_data[column]
        .map(type)
        .value_counts()
    )


Action Obligation
First 20 raw values:
['$701.19 ', '$100,000.00 ', '$400,000.00 ', '$28,511.95 ', '$33,603.43 ', '$48,314.00 ', '$55,056.00 ', '$37,296.00 ', '$47,952.00 ', '$47,952.00 ', '$59,496.00 ', '$0.00 ', '$0.00 ', '$0.00 ', '$0.00 ', '$0.00 ', '$0.00 ', '$0.00 ', '$7,000,000.00 ', '$0.00 ']

Python value types:
Action Obligation
<class 'str'>    150000
Name: count, dtype: int64

Base and All Options Value (Total Contract Value)
First 20 raw values:
['$701.19 ', '$100,000.00 ', '$400,000.00 ', '$28,511.95 ', '$33,603.43 ', '$48,314.00 ', '$55,056.00 ', '$37,296.00 ', '$47,952.00 ', '$47,952.00 ', '$59,496.00 ', '$9,500,000.00 ', '$0.00 ', '$0.00 ', '$0.00 ', '$0.00 ', '$10,500,000.00 ', '$0.00 ', '$7,000,000.00 ', '$0.00 ']

Python value types:
Base and All Options Value (Total Contract Value)
<class 'str'>    150000
Name: count, dtype: int64


In [61]:
for column in numeric_variables:

    raw = model_data[column].astype(str)

    possible_negative = model_data.loc[
        raw.str.contains(
            r'[\(\)-]',
            regex=True,
            na=False
        ),
        column
    ]

    print(f"\n{'=' * 70}")
    print(column)
    print('=' * 70)

    print(
        possible_negative
        .head(20)
        .tolist()
    )


Action Obligation
['($1,631,959.63)', '($15,729.66)', '($11,480.00)', '($412.84)', '($12,502.80)', '($2,184.13)', '($6.95)', '($150.00)', '($3,771.00)', '($20,860.00)', '($24,100.00)', '($50.00)', '($610.00)', '($1,200.00)', '($3,832.63)', '($37,698.94)', '($25,159.20)', '($4,250.00)', '($5,002.24)', '($6,000.00)']

Base and All Options Value (Total Contract Value)
['($48.00)', '($1,631,959.63)', '($15,729.66)', '($11,480.00)', '($412.84)', '($12,502.80)', '($2,184.13)', '($6.95)', '($150.00)', '($3,771.00)', '($20,860.00)', '($24,100.00)', '($50.00)', '($610.00)', '($1,200.00)', '($3,832.63)', '($37,698.94)', '($25,159.20)', '($4,250.00)', '($5,002.24)']


In [63]:
# Function to correct parsing

def parse_currency(series):
    """
    Convert currency-formatted strings to numeric values while preserving
    accounting-style negative values enclosed in parentheses.
    """

    # Work with pandas StringDtype so missing values remain missing
    cleaned = series.astype('string').str.strip()

    # Identify accounting-style negative values BEFORE removing parentheses
    is_negative = (
        cleaned.str.startswith('(')
        & cleaned.str.endswith(')')
    )

    # Remove currency-formatting characters
    cleaned = (
        cleaned
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.replace('(', '', regex=False)
        .str.replace(')', '', regex=False)
        .str.strip()
    )

    # Convert cleaned strings to numeric
    numeric = pd.to_numeric(
        cleaned,
        errors='coerce'
    )

    # Apply negative sign to accounting-style negative values
    numeric = numeric.where(
        ~is_negative,
        -numeric.abs()
    )

    return numeric

In [65]:
# Apply to monetary variables 

for column in numeric_variables:
    model_data[column] = parse_currency(
        model_data[column]
    )

In [67]:
# Validate

for column in numeric_variables:

    print(f"\n{'=' * 70}")
    print(column)
    print('=' * 70)

    print(f"Dtype:    {model_data[column].dtype}")
    print(f"Missing:  {model_data[column].isna().sum():,}")
    print(f"Negative: {(model_data[column] < 0).sum():,}")
    print(f"Zero:     {(model_data[column] == 0).sum():,}")
    print(f"Positive: {(model_data[column] > 0).sum():,}")
    print(f"Minimum:  {model_data[column].min():,.2f}")
    print(f"Maximum:  {model_data[column].max():,.2f}")


Action Obligation
Dtype:    Float64
Missing:  0
Negative: 11,366
Zero:     25,416
Positive: 113,218
Minimum:  -2,747,500,000.50
Maximum:  5,295,000,000.00

Base and All Options Value (Total Contract Value)
Dtype:    Float64
Missing:  0
Negative: 11,452
Zero:     28,372
Positive: 110,176
Minimum:  -3,282,423,246.00
Maximum:  22,992,800,000.00


In [69]:
# Check expected monetary diagnostics established during EDA

expected_diagnostics = {
    'Action Obligation': {
        'missing': 0,
        'negative': 11366,
        'zero': 25416,
        'positive': 113218
    },

    'Base and All Options Value (Total Contract Value)': {
        'missing': 0,
        'negative': 11452,
        'zero': 28372,
        'positive': 110176
    }
}

for column, expected in expected_diagnostics.items():

    actual = {
        'missing': model_data[column].isna().sum(),
        'negative': (model_data[column] < 0).sum(),
        'zero': (model_data[column] == 0).sum(),
        'positive': (model_data[column] > 0).sum()
    }

    assert actual == expected, (
        f"Validation failed for {column}\n"
        f"Expected: {expected}\n"
        f"Actual:   {actual}"
    )

print("All monetary parsing checks passed.")

All monetary parsing checks passed.


In [71]:
# Transform monetary variables using asinh

model_data['Action Obligation Asinh'] = np.arcsinh(
    model_data['Action Obligation']
)

model_data['Total Contract Value Asinh'] = np.arcsinh(
    model_data[
        'Base and All Options Value (Total Contract Value)'
    ]
)

In [73]:
# Validate

monetary_model_features = [
    'Action Obligation Asinh',
    'Total Contract Value Asinh'
]

for column in monetary_model_features:

    print(f"\n{'=' * 70}")
    print(column)
    print('=' * 70)

    print(
        model_data[column]
        .describe(
            percentiles=[
                .01, .05, .25, .50,
                .75, .95, .99
            ]
        )
    )


Action Obligation Asinh
count     150000.0
mean      5.643187
std       5.988419
min     -22.427104
1%      -12.442174
5%       -6.626984
25%       2.423792
50%       6.171705
75%       9.835467
95%      14.149064
99%      16.634459
max      23.083176
Name: Action Obligation Asinh, dtype: Float64

Total Contract Value Asinh
count     150000.0
mean      5.412909
std       6.035206
min     -22.604995
1%      -12.700082
5%       -6.684363
25%            0.0
50%       6.000951
75%       9.490875
95%       14.13048
99%      16.866869
max      24.551594
Name: Total Contract Value Asinh, dtype: Float64


In [75]:
# Verify that zeros and negative values are preserved 

validation_checks = {
    'Action negative signs preserved':
        (
            (model_data['Action Obligation'] < 0)
            ==
            (model_data['Action Obligation Asinh'] < 0)
        ).all(),

    'Action zeros preserved':
        (
            (model_data['Action Obligation'] == 0)
            ==
            (model_data['Action Obligation Asinh'] == 0)
        ).all(),

    'Total value negative signs preserved':
        (
            (
                model_data[
                    'Base and All Options Value (Total Contract Value)'
                ] < 0
            )
            ==
            (
                model_data['Total Contract Value Asinh'] < 0
            )
        ).all(),

    'Total value zeros preserved':
        (
            (
                model_data[
                    'Base and All Options Value (Total Contract Value)'
                ] == 0
            )
            ==
            (
                model_data['Total Contract Value Asinh'] == 0
            )
        ).all()
}

for check, result in validation_checks.items():
    print(f"{check}: {result}")

Action negative signs preserved: True
Action zeros preserved: True
Total value negative signs preserved: True
Total value zeros preserved: True


In [77]:
# Confirm that transformation didn't create missing or infinite values

for column in monetary_model_features:

    print(f"\n{column}")
    print(
        "Missing:",
        model_data[column].isna().sum()
    )

    print(
        "Infinite:",
        np.isinf(model_data[column]).sum()
    )


Action Obligation Asinh
Missing: 0
Infinite: 0

Total Contract Value Asinh
Missing: 0
Infinite: 0


In [79]:
# Freeze categorical variables

categorical_model_features = [
    'Domestic or Foreign Entity',
    'Vendor Address State Name',
    'Solicitation Procedures',
    'Extent Competed',
    'NAICS Code',
    'Type of Contract',
    'Product or Service Code',
    "Contracting Officer's Business Size Determination",
    'Contracting Office Name',
    'Award Type',
    'Funding Department Name',
    'Other Than Full and Open Competition',
    'Fair Opportunity/Limited Sources',
    'Type of Set Aside'
]

In [81]:
# Verify that "missing" category carried over properly

print(
    "Categorical pandas missing values:",
    model_data[categorical_model_features]
    .isna()
    .sum()
    .sum()
)

print(
    "Explicit 'Missing' values:",
    model_data[categorical_model_features]
    .eq('Missing')
    .sum()
    .sum()
)

Categorical pandas missing values: 0
Explicit 'Missing' values: 332808


In [83]:
# Convert all categorical variables to strings

for column in categorical_model_features:
    model_data[column] = (
        model_data[column]
        .astype(str)
    )

In [85]:
# Verify

print(
    model_data[categorical_model_features]
    .dtypes
)

Domestic or Foreign Entity                           object
Vendor Address State Name                            object
Solicitation Procedures                              object
Extent Competed                                      object
NAICS Code                                           object
Type of Contract                                     object
Product or Service Code                              object
Contracting Officer's Business Size Determination    object
Contracting Office Name                              object
Award Type                                           object
Funding Department Name                              object
Other Than Full and Open Competition                 object
Fair Opportunity/Limited Sources                     object
Type of Set Aside                                    object
dtype: object


In [87]:
# Review cardinality of categorical variables

categorical_cardinality = pd.DataFrame({
    'Variable': categorical_model_features,
    'Categories': [
        model_data[column].nunique()
        for column in categorical_model_features
    ]
})

categorical_cardinality = (
    categorical_cardinality
    .sort_values(
        'Categories',
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    categorical_cardinality
    .to_string(index=False)
)

                                         Variable  Categories
                          Contracting Office Name        1454
                          Product or Service Code        1392
                                       NAICS Code         727
                        Vendor Address State Name          57
                          Funding Department Name          57
                                Type of Set Aside          19
             Other Than Full and Open Competition          18
                                 Type of Contract          14
                          Solicitation Procedures          10
                                       Award Type           9
                 Fair Opportunity/Limited Sources           9
                                  Extent Competed           7
                       Domestic or Foreign Entity           6
Contracting Officer's Business Size Determination           2


In [91]:
# Define encoder

categorical_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=True,
    dtype=np.float32
)

In [93]:
# Estimate dimensionality after one-hot encoding

category_counts = (
    model_data[categorical_model_features]
    .nunique()
)

estimated_one_hot_features = category_counts.sum()

print(
    f"Estimated one-hot categorical features: "
    f"{estimated_one_hot_features:,}"
)

Estimated one-hot categorical features: 3,781


In [97]:
# Creat imputer for engineered numeric variables 

numeric_imputer = SimpleImputer(
    strategy='median'
)

In [111]:
# Define feature lists

continuous_model_features = [
    'Action Obligation Asinh',
    'Total Contract Value Asinh',
    'Signing Elapsed Days',
    'Performance Duration Days',
    'Solicitation to Signing Days'
]

bounded_model_features = [
    'Signing Month Sin',
    'Signing Month Cos'
]

binary_model_features = [
    'Solicitation Date Missing',
    'Completion Date Missing',
    'Performance Start Date Missing'
]

In [105]:
# Define preprocessing pipeline

continuous_pipeline = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(strategy='median')
        ),
        (
            'scaler',
            StandardScaler()
        )
    ]
)

In [107]:
# Define categorical encoder

categorical_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=True,
    dtype=np.float32
)

In [113]:
# Define transformer

structured_preprocessor = ColumnTransformer(
    transformers=[
        (
            'continuous',
            continuous_pipeline,
            continuous_model_features
        ),
        (
            'bounded',
            'passthrough',
            bounded_model_features
        ),
        (
            'binary',
            'passthrough',
            binary_model_features
        ),
        (
            'categorical',
            categorical_encoder,
            categorical_model_features
        )
    ],
    remainder='drop'
)

In [115]:
# Define preprocessor

X = structured_preprocessor.fit_transform(model_data)

In [117]:
# Validate model data

print("Rows:", len(model_data))
print("Duplicate index values:", model_data.index.duplicated().sum())

assert len(model_data) == 150_000
assert model_data.index.is_unique

print("Experimental base dataset validated.")

Rows: 150000
Duplicate index values: 0
Experimental base dataset validated.


In [119]:
# Create working reference

experimental_base = model_data.copy(deep=True)

N_OBSERVATIONS = len(experimental_base)
ANOMALY_RATE = 0.03
N_SYNTHETIC_ANOMALIES = int(
    N_OBSERVATIONS * ANOMALY_RATE
)

print(f"Observations: {N_OBSERVATIONS:,}")
print(f"Anomaly rate: {ANOMALY_RATE:.2%}")
print(
    f"Synthetic anomalies per run: "
    f"{N_SYNTHETIC_ANOMALIES:,}"
)

Observations: 150,000
Anomaly rate: 3.00%
Synthetic anomalies per run: 4,500


In [121]:
# Create function to select anomalies

def select_anomaly_indices(
    data,
    anomaly_rate=0.03,
    random_state=None
):
    rng = np.random.default_rng(random_state)

    n_anomalies = int(
        len(data) * anomaly_rate
    )

    anomaly_indices = rng.choice(
        data.index.to_numpy(),
        size=n_anomalies,
        replace=False
    )

    return anomaly_indices

In [123]:
# Test function

test_indices = select_anomaly_indices(
    experimental_base,
    anomaly_rate=ANOMALY_RATE,
    random_state=42
)

print("Selected:", len(test_indices))
print("Unique:", len(np.unique(test_indices)))

assert len(test_indices) == 4_500
assert len(np.unique(test_indices)) == 4_500

Selected: 4500
Unique: 4500


In [125]:
# Establish evaluation labels

def create_ground_truth(
    data,
    anomaly_indices
):
    labels = pd.Series(
        0,
        index=data.index,
        dtype=np.int8,
        name='Synthetic Anomaly'
    )

    labels.loc[anomaly_indices] = 1

    return labels

In [127]:
# Test evaluation labels

test_labels = create_ground_truth(
    experimental_base,
    test_indices
)

print(test_labels.value_counts())
print(test_labels.value_counts(normalize=True))

assert test_labels.sum() == 4_500

Synthetic Anomaly
0    145500
1      4500
Name: count, dtype: int64
Synthetic Anomaly
0    0.97
1    0.03
Name: proportion, dtype: float64
